In [ ]:
# from analyse_results/decode_ana.ipynb
# interim_sum_data/...csv files created in behavior/nlc-model1_post.ipynb

import pandas as pd
import os.path as op
import arviz as az
from stress_risk.utils.data import get_data

bids_folder = '/Users/mrenke/data/ds-stressrisk'
model_folder = op.join(bids_folder, 'derivatives', 'cogmodels')

df = get_data().set_index('group', append=True)
group_mapping = df.groupby(['subject', 'group']).size().reset_index(['group']).drop(0, axis=1)


In [ ]:
def get_subseswise_params(param_posterior, param_name = 'risky_prior_mu'):
    param_ses1_group0 = param_posterior.xs('Intercept', 0, f'{param_name}_regressors')
    param_ses2_group0 = param_posterior.xs('Intercept', 0, f'{param_name}_regressors') + param_posterior.xs('session', 0, f'{param_name}_regressors')
    param_ses1_group1 = param_posterior.xs('Intercept', 0, f'{param_name}_regressors') + param_posterior.xs('group', 0, f'{param_name}_regressors')
    param_ses2_group1 = param_posterior.xs('Intercept', 0, f'{param_name}_regressors') + param_posterior.xs('group', 0, f'{param_name}_regressors') + param_posterior.xs('session', 0, f'{param_name}_regressors') + param_posterior.xs('session:group', 0, f'{param_name}_regressors')

    # only subs of respective groups
    param_ses1_group0 = param_ses1_group0.xs(0, 0, 'group')
    param_ses2_group0 = param_ses2_group0.xs(0, 0, 'group')
    param_ses1_group1 = param_ses1_group1.xs(1, 0, 'group')
    param_ses2_group1 = param_ses2_group1.xs(1, 0, 'group')

    # add session index for merging in next step
    param_ses1_group0['session'] = int(1)
    param_ses2_group0['session'] = int(2)
    param_ses1_group1['session'] = int(1)
    param_ses2_group1['session'] = int(2)

    # merge
    df_param = pd.concat([param_ses1_group0.groupby(['subject']).mean(), param_ses2_group0.groupby(['subject']).mean(), param_ses1_group1.groupby(['subject']).mean(), param_ses2_group1.groupby(['subject']).mean()], axis=0)
    df_param['session'] = df_param['session'].astype(int)
    df_param = df_param.set_index('session', append=True)

    return df_param

In [ ]:
# risky prior

n_model = 1
idata= az.from_netcdf(op.join(model_folder, f'model-{n_model}_trace.netcdf'))

risky_prior_mu_post = idata.posterior['risky_prior_mu'].to_dataframe()
risky_prior_mu_post = risky_prior_mu_post.join(group_mapping).set_index('group', append=True)

In [ ]:
prior_mu_ses1_group0 = risky_prior_mu_post.xs('Intercept', 0, 'risky_prior_mu_regressors')
prior_mu_ses2_group0 = risky_prior_mu_post.xs('Intercept', 0, 'risky_prior_mu_regressors') + risky_prior_mu_post.xs('session', 0, 'risky_prior_mu_regressors')
prior_mu_ses1_group1 = risky_prior_mu_post.xs('Intercept', 0, 'risky_prior_mu_regressors') + risky_prior_mu_post.xs('group', 0, 'risky_prior_mu_regressors')
prior_mu_ses2_group1 = risky_prior_mu_post.xs('Intercept', 0, 'risky_prior_mu_regressors') + risky_prior_mu_post.xs('group', 0, 'risky_prior_mu_regressors') + risky_prior_mu_post.xs('session', 0, 'risky_prior_mu_regressors') + risky_prior_mu_post.xs('session:group', 0, 'risky_prior_mu_regressors')

prior_mu_ses1_group0 = prior_mu_ses1_group0.xs(0, 0, 'group')
prior_mu_ses2_group0 = prior_mu_ses2_group0.xs(0, 0, 'group')

prior_mu_ses1_group1 = prior_mu_ses1_group1.xs(1, 0, 'group')
prior_mu_ses2_group1 = prior_mu_ses2_group1.xs(1, 0, 'group')

In [ ]:
# for completeness also safe prior

safe_prior_mu_post = idata.posterior['safe_prior_mu'].to_dataframe()
safe_prior_mu_post = safe_prior_mu_post.join(group_mapping).set_index('group', append=True)

prior_mu_ses1_group0 = safe_prior_mu_post.xs('Intercept', 0, 'safe_prior_mu_regressors')
prior_mu_ses2_group0 = safe_prior_mu_post.xs('Intercept', 0, 'safe_prior_mu_regressors') + safe_prior_mu_post.xs('session', 0, 'safe_prior_mu_regressors')
prior_mu_ses1_group1 = safe_prior_mu_post.xs('Intercept', 0, 'safe_prior_mu_regressors') + safe_prior_mu_post.xs('group', 0, 'safe_prior_mu_regressors')
prior_mu_ses2_group1 = safe_prior_mu_post.xs('Intercept', 0, 'safe_prior_mu_regressors') + safe_prior_mu_post.xs('group', 0, 'safe_prior_mu_regressors') + safe_prior_mu_post.xs('session', 0, 'safe_prior_mu_regressors') + safe_prior_mu_post.xs('session:group', 0, 'safe_prior_mu_regressors')

prior_mu_ses1_group0 = prior_mu_ses1_group0.xs(0, 0, 'group')
prior_mu_ses2_group0 = prior_mu_ses2_group0.xs(0, 0, 'group')

prior_mu_ses1_group1 = prior_mu_ses1_group1.xs(1, 0, 'group')
prior_mu_ses2_group1 = prior_mu_ses2_group1.xs(1, 0, 'group')

df_change_safe_prior_gr0 = (prior_mu_ses2_group0 - prior_mu_ses1_group0).groupby(['subject']).mean()
df_change_safe_prior_gr1 = (prior_mu_ses2_group1 - prior_mu_ses1_group1).groupby(['subject']).mean()
df_change_safe_prior = pd.concat([df_change_safe_prior_gr0, df_change_safe_prior_gr1], axis=0)
df_change_safe_prior.to_csv('/Users/mrenke/data/ds-stressrisk/interim_sum_data/subwise_change_safe_prior.csv')

In [ ]:
# memory and perceptual noise: model 6

n_model = 6
idata= az.from_netcdf(op.join(model_folder, f'model-{n_model}_trace.netcdf'))

memory_noise_sd = idata.posterior['memory_noise_sd'].to_dataframe() # or idata.posterior['memory_noise_sd'].to_dataframe() ?
percept_noise_sd = idata.posterior['perceptual_noise_sd'].to_dataframe() # or idata.posterior['memory_noise_sd'].to_dataframe() ?

memory_noise_sd = memory_noise_sd.join(group_mapping).set_index('group', append=True)
percept_noise_sd = percept_noise_sd.join(group_mapping).set_index('group', append=True)

mem_noise_ses1_group0 = memory_noise_sd.xs('Intercept', 0, 'memory_noise_sd_regressors')
mem_noise_ses2_group0 = memory_noise_sd.xs('Intercept', 0, 'memory_noise_sd_regressors') + memory_noise_sd.xs('session', 0, 'memory_noise_sd_regressors')
mem_noise_ses1_group1 = memory_noise_sd.xs('Intercept', 0, 'memory_noise_sd_regressors') + memory_noise_sd.xs('group', 0, 'memory_noise_sd_regressors')
mem_noise_ses2_group1 = memory_noise_sd.xs('Intercept', 0, 'memory_noise_sd_regressors') + memory_noise_sd.xs('group', 0, 'memory_noise_sd_regressors') + memory_noise_sd.xs('session', 0, 'memory_noise_sd_regressors') + memory_noise_sd.xs('session:group', 0, 'memory_noise_sd_regressors')

mem_noise_ses1_group0 = mem_noise_ses1_group0.xs(0, 0, 'group')
mem_noise_ses2_group0 = mem_noise_ses2_group0.xs(0, 0, 'group')
mem_noise_ses1_group1 = mem_noise_ses1_group1.xs(1, 0, 'group')
mem_noise_ses2_group1 = mem_noise_ses2_group1.xs(1, 0, 'group')

per_noise_ses1_group0 = percept_noise_sd.xs('Intercept', 0, 'perceptual_noise_sd_regressors')
per_noise_ses2_group0 = percept_noise_sd.xs('Intercept', 0, 'perceptual_noise_sd_regressors') + percept_noise_sd.xs('session', 0, 'perceptual_noise_sd_regressors')
per_noise_ses1_group1 = percept_noise_sd.xs('Intercept', 0, 'perceptual_noise_sd_regressors') + percept_noise_sd.xs('group', 0, 'perceptual_noise_sd_regressors')
per_noise_ses2_group1 = percept_noise_sd.xs('Intercept', 0, 'perceptual_noise_sd_regressors') + percept_noise_sd.xs('group', 0, 'perceptual_noise_sd_regressors') + percept_noise_sd.xs('session', 0, 'perceptual_noise_sd_regressors') + percept_noise_sd.xs('session:group', 0, 'perceptual_noise_sd_regressors')

per_noise_ses1_group0 = per_noise_ses1_group0.xs(0, 0, 'group')
per_noise_ses2_group0 = per_noise_ses2_group0.xs(0, 0, 'group')
per_noise_ses1_group1 = per_noise_ses1_group1.xs(1, 0, 'group')
per_noise_ses2_group1 = per_noise_ses2_group1.xs(1, 0, 'group')

In [ ]:
mem_noise_ses1_group0['session'] = int(1)
mem_noise_ses2_group0['session'] = int(2)
mem_noise_ses1_group1['session'] = int(1)
mem_noise_ses2_group1['session'] = int(2)

df_mem = pd.concat([mem_noise_ses1_group0.groupby(['subject']).mean(), mem_noise_ses2_group0.groupby(['subject']).mean(), mem_noise_ses1_group1.groupby(['subject']).mean(), mem_noise_ses2_group1.groupby(['subject']).mean()], axis=0)
df_mem['session'] = df_mem['session'].astype(int)
df_mem = df_mem.set_index('session', append=True)
per_noise_ses1_group0['session'] = int(1)
per_noise_ses2_group0['session'] = int(2)
per_noise_ses1_group1['session'] = int(1)
per_noise_ses2_group1['session'] = int(2)

df_per = pd.concat([per_noise_ses1_group0.groupby(['subject']).mean(), per_noise_ses2_group0.groupby(['subject']).mean(), per_noise_ses1_group1.groupby(['subject']).mean(), per_noise_ses2_group1.groupby(['subject']).mean()], axis=0)
df_per['session'] = df_per['session'].astype(int)
df_per = df_per.set_index('session', append=True)

df_comb = df_mem.join(df_per)
df_comb.to_csv('/Users/mrenke/data/ds-stressrisk/interim_sum_data/noises_model-6.csv')

In [ ]:
# n2/n1 evidence, all except model-6

In [ ]:


n2_evidence_sd = idata.posterior['n2_evidence_sd'].to_dataframe()
n2_evidence_sd = n2_evidence_sd.join(group_mapping).set_index('group', append=True)

n2_esmu_ses1_group0 = n2_evidence_sd.xs('Intercept', 0, 'n2_evidence_sd_regressors')
n2_esmu_ses2_group0 = n2_evidence_sd.xs('Intercept', 0, 'n2_evidence_sd_regressors') + n2_evidence_sd.xs('session', 0, 'n2_evidence_sd_regressors')
n2_esmu_ses1_group1 = n2_evidence_sd.xs('Intercept', 0, 'n2_evidence_sd_regressors') + n2_evidence_sd.xs('group', 0, 'n2_evidence_sd_regressors')
n2_esmu_ses2_group1 = n2_evidence_sd.xs('Intercept', 0, 'n2_evidence_sd_regressors') + n2_evidence_sd.xs('group', 0, 'n2_evidence_sd_regressors') + n2_evidence_sd.xs('session', 0, 'n2_evidence_sd_regressors') + n2_evidence_sd.xs('session:group', 0, 'n2_evidence_sd_regressors')

n2_esmu_ses1_group0 = n2_esmu_ses1_group0.xs(0, 0, 'group')
n2_esmu_ses2_group0 = n2_esmu_ses2_group0.xs(0, 0, 'group')
n2_esmu_ses1_group1 = n2_esmu_ses1_group1.xs(1, 0, 'group')
n2_esmu_ses2_group1 = n2_esmu_ses2_group1.xs(1, 0, 'group')
n2_esmu_ses1_group0['session'] = int(1)
n2_esmu_ses2_group0['session'] = int(2)
n2_esmu_ses1_group1['session'] = int(1)
n2_esmu_ses2_group1['session'] = int(2)

df_n2_esmu = pd.concat([n2_esmu_ses1_group0.groupby(['subject']).mean(), n2_esmu_ses2_group0.groupby(['subject']).mean(), n2_esmu_ses1_group1.groupby(['subject']).mean(), n2_esmu_ses2_group1.groupby(['subject']).mean()], axis=0)
df_n2_esmu['n2_evidence_sd'] = np.exp(df_n2_esmu['n2_evidence_sd'])
df_n2_esmu['session'] = df_n2_esmu['session'].astype(int)

df_n2_esmu.to_csv('/Users/mrenke/data/ds-stressrisk/interim_sum_data/subwise_seswise_n2_evidence_sd.csv')